# 08 — Model evaluation and locked selection
Compare all three saved models on identical frozen rows, verify the original validation-only choice and demonstrate the future dashboard prediction contract. This is reproducibility verification, not retraining or a new selection experiment.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.features import FEATURE_COLUMNS
from ml.phase4_models import MODEL_NAMES, load_predictor, metrics, numeric_matrix, predictions_agree, select_model, sha256
from ml.train_phase4 import read_partition, verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
results = json.loads(Path('ml/artifacts/phase4/model_results.json').read_text(encoding='utf-8'))
lock_path = Path('ml/artifacts/phase4/validation_selection.json')
lock = json.loads(lock_path.read_text(encoding='utf-8'))
verify_manifest(manifest)
lock_hash = sha256(lock_path)
assert lock_hash == results['selection_lock_sha256']
assert lock['test_opened'] is False and results['test_used_for_selection'] is False
bundle = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
print('Interpreter:', sys.executable)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe


## Fair comparison and fixed decision rule
Read one partition at a time and give every model the exact same feature matrix and targets. Check the recorded timestamp-row identity. Compute MAE, RMSE and R² from predictions clipped to [0,12], without rounding. Verify raw metrics and clipping counts too.
Reconfirm the pre-existing selection using validation only. Test rows were already evaluated in Phase 4; this notebook recomputes their scores for reproducibility after verifying that the original choice is locked. It never tunes against test results.

In [2]:
comparison = []
validation_scores = {}
for split in ('validation', 'test'):
    x, y, identity = read_partition(manifest, split)
    assert identity == results['partitions'][split]
    for name in MODEL_NAMES:
        measured = metrics(y, bundle.models[name].predict(x))
        expected = results[split][name]
        assert measured['rows'] == len(y) == expected['rows']
        assert measured['clipped_count'] == expected['clipped_count']
        for kind in ('raw', 'clipped'):
            for metric in ('mae', 'rmse', 'r2'):
                np.testing.assert_allclose(measured[kind][metric], expected[kind][metric], rtol=0, atol=1e-12)
        comparison.append({'split': split, 'model': name, 'rows': len(y), **measured['clipped'], 'clipped_count': measured['clipped_count'], 'origin_sha256': identity['origin_sha256']})
        if split == 'validation':
            validation_scores[name] = measured
    if split == 'validation':
        assert select_model(validation_scores) == lock['selected_model'] == bundle.selected_model == 'persistence'
        print('PASS: original validation-only selection reconfirmed before test recomputation')
table = pd.DataFrame(comparison)
assert table.groupby('split').origin_sha256.nunique().eq(1).all()
assert table.groupby('split').rows.nunique().eq(1).all()
display(table.drop(columns='origin_sha256'))
baseline = table.loc[table.model.eq('persistence')].set_index('split').mae
table['mae_reduction_vs_persistence'] = [float(baseline.loc[row.split] - row.mae) for row in table.itertuples()]
display(table[['split', 'model', 'mae_reduction_vs_persistence']])
assert sha256(lock_path) == lock_hash
print('PASS: all six score sets agree with frozen results; selection lock unchanged')

PASS: original validation-only selection reconfirmed before test recomputation


,split,model,rows,mae,rmse,r2,clipped_count
0,validation,persistence,10577,0.339983,0.684103,0.950447,0
1,validation,linear_regression,10577,0.414362,0.656363,0.954385,1302
2,validation,random_forest,10577,0.436999,0.680876,0.950914,0
3,test,persistence,10578,0.307714,0.648092,0.945254,0
4,test,linear_regression,10578,0.373729,0.613500,0.950942,1590
5,test,random_forest,10578,0.394675,0.641002,0.946445,0


,split,model,mae_reduction_vs_persistence
0,validation,persistence,0.000000
1,validation,linear_regression,-0.074379
2,validation,random_forest,-0.097016
3,test,persistence,0.000000
4,test,linear_regression,-0.066015
5,test,random_forest,-0.086961


PASS: all six score sets agree with frozen results; selection lock unchanged


## Simple prediction interface for future integration
Load the trusted checksum-verified bundle once, supply a named DataFrame or feature dictionary with exactly the frozen 37 numeric features, then call `bundle.predict(features)`. Default inference uses the locked selected model. Optional `model_name` supports comparisons. Output is a finite one-dimensional fractional occupancy forecast in [0,12], with a 15-minute horizon.
A live caller will need a complete causal seven-day history, matching local calendar/timezone semantics, and an explicit policy for unavailable/stale measurements. This example uses a frozen test feature row only; it does not connect to a dashboard or present the result as a live forecast.

In [3]:
sample = pd.DataFrame(x[:3], columns=FEATURE_COLUMNS, copy=False)
forecast = bundle.predict(sample)
record = {name: float(sample.iloc[0][name]) for name in FEATURE_COLUMNS}
assert predictions_agree(bundle.predict(record), forecast[:1])
reloaded = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
for name in MODEL_NAMES:
    assert predictions_agree(bundle.predict(sample, name), reloaded.predict(sample, name))
    assert ((reloaded.predict(sample, name) >= 0) & (reloaded.predict(sample, name) <= 12)).all()
try:
    numeric_matrix({key: value for key, value in record.items() if key != 'P12'})
except ValueError as error:
    print('Expected missing-feature rejection:', error)
else:
    raise AssertionError('Incomplete feature record was accepted')
display(pd.DataFrame({'forecast_occupied_t_plus_15': forecast}))
print('PASS: reload, dictionary/DataFrame agreement, bounds and missing-feature rejection')

Expected missing-feature rejection: Provide one or more records with exactly FEATURE_COLUMNS


,forecast_occupied_t_plus_15
0,3.0
1,3.0
2,3.0


PASS: reload, dictionary/DataFrame agreement, bounds and missing-feature rejection


In [4]:
from pathlib import Path
storage = sum(p.stat().st_size for folder in ('ml/data', 'ml/artifacts') for p in Path(folder).rglob('*') if p.is_file())
assert storage < 50_000_000
assert sha256(results['artifact']['path']) == results['artifact']['sha256']
assert results['models_refitted_on_validation'] is False
acceptance = {'frozen_dataset_checksums': True, 'same_held_out_rows_for_all_models': True, 'six_metric_sets_reproduced': True, 'original_validation_selection_preserved': True, 'trusted_bundle_reload': True, 'bounded_prediction_interface': True, 'storage_under_50_MB': True}
display(pd.Series(acceptance, name='checks_executed_above').to_frame())
print('Data/model bytes:', storage)
print('Selected:', bundle.selected_model, '| horizon:', bundle.horizon_minutes, '| source:', bundle.data_source)

,checks_executed_above
frozen_dataset_checksums,True
same_held_out_rows_for_all_models,True
six_metric_sets_reproduced,True
original_validation_selection_preserved,True
trusted_bundle_reload,True
bounded_prediction_interface,True
storage_under_50_MB,True


Data/model bytes: 8190119
Selected: persistence | horizon: 15 | source: synthetic


## Final interpretation
Persistence remains the validation-MAE winner. A negative MAE reduction means the competing model performs worse on that partition. Better Linear Regression RMSE does not alter the declared MAE-first rule.
All outputs are recomputations on frozen **synthetic** observations. There is no new real-world accuracy claim, firmware verification, API integration or dashboard implementation. The original Phase 3/4 PDFs, source, datasets and model artifacts must remain unchanged. Consult `reports/evidence/notebook_execution_verification.json` for actual notebook execution status, environment and preservation checks.